# codex_project / Ditto Data Pipeline

This notebook is the Colab entrypoint for the current `data_pipeline/` codebase. It processes YouTube videos and PDF books, enriches metadata with an LLM, creates semantic chunks, generates BGE-M3 embeddings, and uploads vectors to Qdrant.

Run CPU cells first. Switch to a T4 GPU runtime before the embedding cell if you want faster vector generation.

## What changed from the older notebook

- Video chunks now live in `data_pipeline/videos/processed_chunks/`, not `enriched_json/`.
- Video chunk text is stored as `text`, not `marathi_raw`.
- The old `TranscriptProcessor` class call is replaced by `data_pipeline.videos.transcript_processor.process_directory()`.
- Dynamo uploaders are now `data_pipeline.videos.dynamo_uploader` and `data_pipeline.books.dynamo_uploader`.
- Old cleanup/fix scripts under `scripts/metadata` and `scripts/qdrant` are not assumed to exist. Qdrant upload is implemented directly in this notebook.
- The notebook symlinks Google Drive into the current folder structure, including `processed_chunks` for videos.

In [ ]:
# Cell 1: Mount Google Drive and create persistent folders
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/codex_project')
DRIVE_DIRS = {
    'video_output': DRIVE_ROOT / 'output',
    'video_meta': DRIVE_ROOT / 'enriched_metadata',
    'video_chunks': DRIVE_ROOT / 'processed_chunks',
    'book_input': DRIVE_ROOT / 'input_books',
    'book_output': DRIVE_ROOT / 'books_output',
    'book_meta': DRIVE_ROOT / 'books_enriched_metadata',
    'book_chunks': DRIVE_ROOT / 'processed_books_chunks',
    'video_embedded': DRIVE_ROOT / 'embedded_video_chunks',
    'book_embedded': DRIVE_ROOT / 'embedded_book_chunks',
    'qdrant_artifacts': DRIVE_ROOT / 'qdrant_artifacts',
}

for directory in DRIVE_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

print(f'Drive storage ready: {DRIVE_ROOT}')

In [ ]:
# Cell 2: Clone repository, install dependencies, and link Drive folders
from google.colab import userdata
from pathlib import Path
import os
import shutil
import subprocess

def get_secret(name, default=None, required=False):
    try:
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f'Missing required Colab secret: {name}') from exc
        return default
    return value if value not in (None, '') else default

REPO_DIR = Path('/content/repo')
DEFAULT_REPO_URL = 'https://github.com/chanekarayush/codex-hackathon-chanekarayush.git'
REPO_URL = get_secret('GITHUB_REPO_URL', DEFAULT_REPO_URL)
REPO_BRANCH = get_secret('GITHUB_BRANCH', 'feat/data-processing')
GITHUB_TOKEN = get_secret('GITHUB_TOKEN')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_url = REPO_URL
if GITHUB_TOKEN and REPO_URL.startswith('https://github.com/'):
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@', 1)

subprocess.run([
    'git', 'clone', '--quiet',
    '--branch', REPO_BRANCH,
    '--single-branch',
    clone_url,
    str(REPO_DIR),
], check=True)
print(f'Cloned {REPO_URL} branch {REPO_BRANCH}')

links = {
    REPO_DIR / 'data_pipeline/videos/output': DRIVE_DIRS['video_output'],
    REPO_DIR / 'data_pipeline/videos/enriched_metadata': DRIVE_DIRS['video_meta'],
    REPO_DIR / 'data_pipeline/videos/processed_chunks': DRIVE_DIRS['video_chunks'],
    REPO_DIR / 'data_pipeline/books/input_books': DRIVE_DIRS['book_input'],
    REPO_DIR / 'data_pipeline/books/books_output': DRIVE_DIRS['book_output'],
    REPO_DIR / 'data_pipeline/books/books_enriched_metadata': DRIVE_DIRS['book_meta'],
    REPO_DIR / 'data_pipeline/books/processed_books_chunks': DRIVE_DIRS['book_chunks'],
}

for link_path, target_path in links.items():
    if link_path.is_symlink() or link_path.exists():
        if link_path.is_dir() and not link_path.is_symlink():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()
    link_path.parent.mkdir(parents=True, exist_ok=True)
    link_path.symlink_to(target_path, target_is_directory=True)

%cd /content/repo
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-mar
!pip install -q -r requirements.txt
!pip install -q -e /content/repo

print('Repository cloned, dependencies installed, and Drive symlinks created.')

In [ ]:
# Cell 3: Load API keys and runtime configuration
from google.colab import userdata
import os

def get_secret(name, default=None, required=False):
    try:
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f'Missing required Colab secret: {name}') from exc
        return default
    return value if value not in (None, '') else default

def set_secret_env(env_name, secret_name=None, required=False):
    secret_name = secret_name or env_name
    value = get_secret(secret_name, required=required)
    if value:
        os.environ[env_name] = value
    return value

os.environ['DITTO_LLM_MODEL'] = get_secret('DITTO_LLM_MODEL', 'gpt-4o-mini')
set_secret_env('OPENAI_API_KEY')
set_secret_env('DITTO_LLM_TEMPERATURE')
set_secret_env('DITTO_LLM_MAX_ATTEMPTS')

set_secret_env('QDRANT_URL')
set_secret_env('QDRANT_API_KEY')
set_secret_env('AWS_ACCESS_KEY_ID')
set_secret_env('AWS_SECRET_ACCESS_KEY')
set_secret_env('AWS_DEFAULT_REGION')

VIDEO_URLS = [
    # 'https://www.youtube.com/watch?v=VIDEO_ID',
]

RUN_VIDEOS = True
RUN_BOOKS = True
RUN_LLM_ENRICHMENT = True
RUN_DYNAMO_UPLOAD = False
RUN_EMBEDDINGS = True
RUN_QDRANT_UPLOAD = False

if RUN_LLM_ENRICHMENT and not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY is required when RUN_LLM_ENRICHMENT=True.')

VIDEO_DYNAMO_TABLE = get_secret('DITTO_VIDEOS_TABLE', 'codex_project-content')
VIDEO_SEGMENTS_DYNAMO_TABLE = get_secret('DITTO_VIDEO_SEGMENTS_TABLE')
BOOK_DYNAMO_TABLE = get_secret('DITTO_BOOKS_TABLE', 'codex_project-books')

VIDEO_QDRANT_COLLECTION = get_secret('VIDEO_QDRANT_COLLECTION', 'codex_project-videos')
BOOK_QDRANT_COLLECTION = get_secret('BOOK_QDRANT_COLLECTION', 'codex_project-books')
RECREATE_QDRANT_COLLECTIONS = False

print('Runtime configured.')
print(f"OpenAI model: {os.environ['DITTO_LLM_MODEL']}")

## CPU phase: extraction, enrichment, and chunking

For books, upload PDFs into `codex_project/input_books` on Google Drive before running this section.

In [ ]:
# Cell 4: Run video extraction, enrichment, and zero-drift chunking
import sys
sys.path.insert(0, '/content/repo')

from data_pipeline.videos.main import ingest_urls
from data_pipeline.videos.video_enricher import enrich_directory as enrich_videos
from data_pipeline.videos.transcript_processor import process_directory as chunk_videos

if RUN_VIDEOS:
    if VIDEO_URLS:
        ingest_urls(VIDEO_URLS)
    else:
        print('VIDEO_URLS is empty; skipping transcript download and using existing Drive transcripts.')

    if RUN_LLM_ENRICHMENT:
        enrich_videos()
    else:
        print('Skipping video LLM enrichment.')

    chunk_videos()
else:
    print('RUN_VIDEOS is False; skipped video pipeline.')

In [ ]:
# Cell 5: Run book extraction, enrichment, and page-aware chunking
from data_pipeline.books.books_main import run_book_pipeline

if RUN_BOOKS:
    run_book_pipeline(skip_enrichment=not RUN_LLM_ENRICHMENT)
else:
    print('RUN_BOOKS is False; skipped book pipeline.')

In [ ]:
# Cell 6: Optional DynamoDB metadata upload
if RUN_DYNAMO_UPLOAD:
    from data_pipeline.videos.dynamo_uploader import upload_directory as upload_video_metadata
    from data_pipeline.books.dynamo_uploader import upload_directory as upload_book_metadata

    upload_video_metadata(
        videos_table_name=VIDEO_DYNAMO_TABLE,
        segments_table_name=VIDEO_SEGMENTS_DYNAMO_TABLE,
    )
    upload_book_metadata(table_name=BOOK_DYNAMO_TABLE)
else:
    print('RUN_DYNAMO_UPLOAD is False; skipped DynamoDB upload.')

## GPU phase: embeddings and Qdrant upload

Switch to a T4 GPU runtime for this section. CPU works, but BGE-M3 will be slow.

In [ ]:
# Cell 7: Load BGE-M3 embedding model
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
embedder = SentenceTransformer('BAAI/bge-m3', device=device)
print('Embedder ready.')

In [ ]:
# Cell 8: Embed video and book chunks into Drive-backed JSON files
from pathlib import Path
import glob
import json
import os
import tempfile

BATCH_SIZE = 32

def atomic_json_write(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', delete=False, dir=path.parent, suffix='.tmp') as tmp:
        json.dump(payload, tmp, ensure_ascii=False, indent=2)
        tmp.write('\n')
        tmp_path = Path(tmp.name)
    tmp_path.replace(path)

def embed_chunk_file(input_path, output_path):
    input_path = Path(input_path)
    output_path = Path(output_path)
    if output_path.exists():
        print(f'Skipping existing embeddings: {output_path.name}')
        return {'status': 'skipped', 'count': 0}

    chunks = json.loads(input_path.read_text(encoding='utf-8'))
    if not chunks:
        print(f'Skipping empty chunk file: {input_path.name}')
        return {'status': 'empty', 'count': 0}

    texts = [str(chunk.get('text') or '').strip() for chunk in chunks]
    vectors = embedder.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    embedded = []
    for chunk, vector in zip(chunks, vectors):
        record = dict(chunk)
        record['embedding_vector'] = vector.tolist()
        embedded.append(record)

    atomic_json_write(output_path, embedded)
    print(f'Embedded {len(embedded)} chunks -> {output_path.name}')
    return {'status': 'processed', 'count': len(embedded)}

def embed_directory(input_glob, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    stats = {'processed': 0, 'skipped': 0, 'empty': 0, 'chunks': 0}
    for input_file in sorted(glob.glob(input_glob)):
        input_path = Path(input_file)
        output_name = input_path.name.replace('_chunks.json', '_embedded.json')
        result = embed_chunk_file(input_path, output_dir / output_name)
        stats[result['status']] += 1
        stats['chunks'] += result['count']
    return stats

if RUN_EMBEDDINGS:
    video_stats = embed_directory('/content/repo/data_pipeline/videos/processed_chunks/*_chunks.json', DRIVE_DIRS['video_embedded'])
    book_stats = embed_directory('/content/repo/data_pipeline/books/processed_books_chunks/*_chunks.json', DRIVE_DIRS['book_embedded'])
    print({'video': video_stats, 'book': book_stats})
else:
    print('RUN_EMBEDDINGS is False; skipped embeddings.')

In [ ]:
# Cell 9: Upload embedded chunks to Qdrant with dense + BM25 sparse vectors
from collections import Counter
from pathlib import Path
import glob
import json
import math
import os
import re
import uuid

TOKEN_PATTERN = re.compile(r'[\w\u0900-\u097F]+', flags=re.UNICODE)

def tokenize(text):
    return [token.lower() for token in TOKEN_PATTERN.findall(text or '')]

def load_embedded_records(pattern, content_type):
    records = []
    for path in sorted(glob.glob(str(pattern))):
        for record in json.loads(Path(path).read_text(encoding='utf-8')):
            if record.get('embedding_vector') and record.get('text'):
                item = dict(record)
                item['content_type'] = content_type
                records.append(item)
    return records

def build_bm25(records, artifact_path, k1=1.5, b=0.75):
    tokenized = [tokenize(record.get('text', '')) for record in records]
    vocab = {}
    dfs = Counter()
    for tokens in tokenized:
        unique_tokens = set(tokens)
        for token in unique_tokens:
            if token not in vocab:
                vocab[token] = len(vocab)
            dfs[token] += 1

    doc_count = max(len(records), 1)
    avgdl = sum(len(tokens) for tokens in tokenized) / max(len(tokenized), 1)
    idf = {token: math.log(1 + ((doc_count - df + 0.5) / (df + 0.5))) for token, df in dfs.items()}

    artifact = {'vocab': vocab, 'idf': idf, 'avgdl': avgdl, 'k1': k1, 'b': b}
    atomic_json_write(artifact_path, artifact)

    sparse_vectors = []
    for tokens in tokenized:
        counts = Counter(tokens)
        doc_len = len(tokens) or 1
        indices = []
        values = []
        for token, tf in counts.items():
            denominator = tf + k1 * (1 - b + b * (doc_len / max(avgdl, 1e-9)))
            score = idf[token] * ((tf * (k1 + 1)) / denominator)
            indices.append(vocab[token])
            values.append(float(score))
        sparse_vectors.append((indices, values))
    return sparse_vectors

def qdrant_collection_exists(client, collection_name):
    try:
        client.get_collection(collection_name)
        return True
    except Exception:
        return False

def ensure_qdrant_collection(client, collection_name, dense_size, recreate=False):
    from qdrant_client import models

    exists = qdrant_collection_exists(client, collection_name)
    if exists and recreate:
        client.delete_collection(collection_name=collection_name)
        exists = False
    if not exists:
        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                'dense': models.VectorParams(size=dense_size, distance=models.Distance.COSINE),
            },
            sparse_vectors_config={
                'sparse': models.SparseVectorParams(index=models.SparseIndexParams(on_disk=False)),
            },
        )

def upload_records_to_qdrant(collection_name, records, artifact_name):
    if not records:
        print(f'No records to upload for {collection_name}.')
        return

    from qdrant_client import QdrantClient, models

    client = QdrantClient(
        url=os.environ['QDRANT_URL'],
        api_key=os.environ.get('QDRANT_API_KEY'),
        timeout=120,
    )
    dense_size = len(records[0]['embedding_vector'])
    ensure_qdrant_collection(client, collection_name, dense_size, recreate=RECREATE_QDRANT_COLLECTIONS)

    sparse_vectors = build_bm25(records, DRIVE_DIRS['qdrant_artifacts'] / artifact_name)
    batch = []
    total = 0
    for record, sparse in zip(records, sparse_vectors):
        payload = {key: value for key, value in record.items() if key != 'embedding_vector'}
        stable_key = payload.get('chunk_id') or f"{payload.get('content_type')}:{payload.get('start_index')}:{payload.get('text', '')[:80]}"
        point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'{collection_name}:{stable_key}'))
        batch.append(
            models.PointStruct(
                id=point_id,
                vector={
                    'dense': record['embedding_vector'],
                    'sparse': models.SparseVector(indices=sparse[0], values=sparse[1]),
                },
                payload=payload,
            )
        )
        if len(batch) >= 64:
            client.upsert(collection_name=collection_name, points=batch, wait=True)
            total += len(batch)
            batch = []
    if batch:
        client.upsert(collection_name=collection_name, points=batch, wait=True)
        total += len(batch)
    print(f'Uploaded {total} points to {collection_name}.')

if RUN_QDRANT_UPLOAD:
    if not os.environ.get('QDRANT_URL'):
        raise RuntimeError('QDRANT_URL is required for Qdrant upload.')
    video_records = load_embedded_records(DRIVE_DIRS['video_embedded'] / '*_embedded.json', 'youtube_video')
    book_records = load_embedded_records(DRIVE_DIRS['book_embedded'] / '*_embedded.json', 'pdf_book')
    upload_records_to_qdrant(VIDEO_QDRANT_COLLECTION, video_records, 'videos_bm25_vocab_idf.json')
    upload_records_to_qdrant(BOOK_QDRANT_COLLECTION, book_records, 'books_bm25_vocab_idf.json')
else:
    print('RUN_QDRANT_UPLOAD is False; skipped Qdrant upload.')

In [ ]:
# Cell 10: Verify output counts
from pathlib import Path

verification_dirs = {
    'Raw video transcripts': DRIVE_DIRS['video_output'],
    'Video metadata': DRIVE_DIRS['video_meta'],
    'Video chunks': DRIVE_DIRS['video_chunks'],
    'Embedded video chunks': DRIVE_DIRS['video_embedded'],
    'Input PDFs': DRIVE_DIRS['book_input'],
    'Book page JSON': DRIVE_DIRS['book_output'],
    'Book metadata': DRIVE_DIRS['book_meta'],
    'Book chunks': DRIVE_DIRS['book_chunks'],
    'Embedded book chunks': DRIVE_DIRS['book_embedded'],
    'Qdrant artifacts': DRIVE_DIRS['qdrant_artifacts'],
}

print('=' * 64)
print('Pipeline output verification')
print('=' * 64)
for label, directory in verification_dirs.items():
    files = [path for path in Path(directory).glob('*') if path.is_file()]
    print(f'{label:<28} {len(files):>5} files  {directory}')
print('=' * 64)